In [5]:
# ==========================================
# תא (Cell) מספר 1: ייבוא ספריות, פונקציית העיבוד וקריאת נתונים
# ==========================================
import pandas as pd
import numpy as np
import re

def preprocess_movie_data(df):
    """
    פונקציה דטרמיניסטית לעיבוד וניקוי נתונים לפני הכנסה למודל.
    מכינה את התקציב, השנים, השפות, המדינות והז'אנרים.
    לא משלימה ערכים חסרים (NaN) כדי למנוע זליגת נתונים (Data Leakage)!
    """
    # יצירת עותק כדי לא לדרוס את הדאטה המקורי
    df_clean = df.copy()

    # 1. טיפול בעשור
    if 'startYear' in df_clean.columns:
        df_clean['startYear'] = pd.to_numeric(df_clean['startYear'], errors='coerce')
        decades = [1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
        for dec in decades:
            df_clean[f'is_decade_{dec}'] = ((df_clean['startYear'] // 10) * 10 == dec).astype(int)
        df_clean['is_decade_unknown'] = df_clean['startYear'].isna().astype(int)

    # 2. טיפול בתקציב (ללא השלמה - רק ניקוי, חוק ה-1000 ולוגריתם)
    def parse_financial_value(val):
        if pd.isna(val) or str(val).strip() == '' or str(val).lower() == 'unknown':
            return np.nan
        val_str = str(val).lower()
        val_str = re.sub(r'[$,£€¥]', '', val_str).replace(',', '')
        
        multiplier, has_explicit = 1, False
        if 'billion' in val_str or 'b' in val_str:
            multiplier, has_explicit = 1000000000, True
        elif 'million' in val_str or 'm' in val_str:
            multiplier, has_explicit = 1000000, True
        elif 'thousand' in val_str or 'k' in val_str:
            multiplier, has_explicit = 1000, True
            
        match = re.search(r'\d+(\.\d+)?', val_str)
        if match:
            result = float(match.group()) * multiplier
            if result == 0: return np.nan
            if not has_explicit:
                if 0 < result <= 100: result *= 1000000  
                elif 100 < result <= 1000: return np.nan 
            return result
        return np.nan

    if 'budget' in df_clean.columns:
        df_clean['budget_parsed'] = df_clean['budget'].apply(parse_financial_value)
        df_clean['is_budget_missing'] = df_clean['budget_parsed'].isna().astype(int)
        df_clean['budget_log'] = np.log1p(df_clean['budget_parsed'])

    # 3. שפות
    if 'Language' in df_clean.columns:
        top_langs = ['English', 'French', 'Hindi', 'Spanish', 'Italian', 'Japanese', 'Tamil', 'German', 'Telugu', 'Malayalam']
        def cat_lang(val):
            res = {f'is_{l}': 0 for l in top_langs}
            res.update({'non_primary_language': 0, 'unknown_language': 0})
            val = str(val).strip()
            if val in ['Not Found', 'unknown', 'nan'] or len(val) > 80:
                res['unknown_language'] = 1
                return pd.Series(res)
            text = val.lower()
            for l in top_langs:
                if l.lower() in text:
                    res[f'is_{l}'] = 1
                    text = text.replace(l.lower(), '')
            if len(re.sub(r'[^\w\s]', '', text).strip()) > 2: res['non_primary_language'] = 1
            if sum(res.values()) == 0: res['unknown_language'] = 1
            return pd.Series(res)
        df_clean = pd.concat([df_clean, df_clean['Language'].apply(cat_lang)], axis=1)

    # 4. מדינות
    if 'Country' in df_clean.columns:
        top_countries = ['United States', 'United Kingdom', 'India', 'France', 'Japan', 'Canada', 'Germany', 'Italy', 'Spain', 'Australia']
        def cat_country(val):
            res = {f'is_country_{c.replace(" ", "_")}': 0 for c in top_countries}
            res.update({'is_other_country': 0, 'unknown_country': 0})
            val = str(val).strip()
            if val in ['Not Found', 'unknown', 'nan'] or len(val) > 80:
                res['unknown_country'] = 1
                return pd.Series(res)
            text = val.lower()
            for c in top_countries:
                if c.lower() in text:
                    res[f'is_country_{c.replace(" ", "_")}'] = 1
                    text = text.replace(c.lower(), '')
            if len(re.sub(r'[^\w\s]', '', text).strip()) > 2: res['is_other_country'] = 1
            if sum(res.values()) == 0: res['unknown_country'] = 1
            return pd.Series(res)
        df_clean = pd.concat([df_clean, df_clean['Country'].apply(cat_country)], axis=1)

    # 5. ז'אנרים
    if 'genres' in df_clean.columns:
        top_genres = ['Drama', 'Comedy', 'Romance', 'Action', 'Documentary', 'Crime', 'Thriller', 'Horror', 'Adventure', 'Mystery']
        def cat_genre(val):
            res = {f'is_genre_{g}': 0 for g in top_genres}
            res.update({'is_other_genre': 0, 'unknown_genre': 0})
            val = str(val).strip()
            if val in ['Not Found', 'unknown', '\\N', 'nan'] or len(val) > 80:
                res['unknown_genre'] = 1
                return pd.Series(res)
            text = val.replace('[','').replace(']','').replace("'",'').replace('"','').lower()
            for g in top_genres:
                if g.lower() in text:
                    res[f'is_genre_{g}'] = 1
                    text = text.replace(g.lower(), '')
            if len(re.sub(r'[^\w\s]', '', text).strip()) > 2: res['is_other_genre'] = 1
            if sum(res.values()) == 0: res['unknown_genre'] = 1
            return pd.Series(res)
        df_clean = pd.concat([df_clean, df_clean['genres'].apply(cat_genre)], axis=1)

    # 6. זמן ריצה ודירוג (המרה למספרים)
    if 'averageRating' in df_clean.columns:
        df_clean['averageRating'] = pd.to_numeric(df_clean['averageRating'], errors='coerce')
    if 'runtimeMinutes' in df_clean.columns:
        df_clean['runtimeMinutes'] = pd.to_numeric(df_clean['runtimeMinutes'], errors='coerce')

    # 7. הסרת עמודות מיותרות או טקסטואליות
    cols_to_drop = [
        'tconst', 'primaryTitle', 'plot', 'BoxOffice', 'numVotes',
        'budget', 'budget_parsed', 'startYear', 'Language', 'Country', 'genres'
    ]
    df_model = df_clean.drop(columns=[col for col in cols_to_drop if col in df_clean.columns])

    return df_model


# --- טעינת הנתונים והפעלת הפונקציה ---
print("טוען את קובץ הנתונים הגולמי (Raw Dataset)...")
raw_df = pd.read_csv(r"C:\Users\yarin\Downloads\dataset.csv", low_memory=False)

print("מעביר את הנתונים דרך פונקציית העיבוד המקדימה...")
processed_df = preprocess_movie_data(raw_df)

print(f"העיבוד הושלם! גודל הדאטה המעובד: {processed_df.shape}")

טוען את קובץ הנתונים הגולמי (Raw Dataset)...
מעביר את הנתונים דרך פונקציית העיבוד המקדימה...
העיבוד הושלם! גודל הדאטה המעובד: (133884, 54)


In [7]:
# ==========================================
# חלק 2: מחלקות חכמות, Pipeline ואימון מודל
# ==========================================
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# --- א. יצירת רכיבים מותאמים אישית (Custom Transformers) ---

class PolynomialFeaturesGenerator(BaseEstimator, TransformerMixin):
    """ מוסיף זמן ריצה בריבוע """
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_new = X.copy()
        if isinstance(X_new, pd.DataFrame) and 'runtimeMinutes' in X_new.columns:
            X_new['runtimeMinutes_sq'] = X_new['runtimeMinutes'] ** 2
        return X_new

class CastTargetEncoder(BaseEstimator, TransformerMixin):
    """
    מקודד שחקנים עם סינון נוקשה (min 5) והחלקה בייסיאנית.
    מונע Data Leakage לחלוטין.
    """
    def __init__(self, min_count=5, smoothing_m=30):
        self.min_count = min_count
        self.smoothing_m = smoothing_m
        self.actor_means_ = {}
        self.global_mean_ = 0

    def fit(self, X, y):
        df_temp = pd.DataFrame({'actors': X['lead_actors_ids'].astype(str), 'rating': y})
        df_temp['actors'] = df_temp['actors'].apply(lambda x: [a.strip() for a in x.split(',') if a.strip() not in ['', 'nan']])
        df_exploded = df_temp.explode('actors').dropna(subset=['actors'])
        
        self.global_mean_ = y.mean()
        
        stats = df_exploded.groupby('actors')['rating'].agg(['mean', 'count'])
        # סינון - רק שחקנים עם לפחות X סרטים
        stats = stats[stats['count'] >= self.min_count]
        
        # נוסחת ההחלקה
        stats['smoothed'] = ((stats['count'] * stats['mean']) + (self.smoothing_m * self.global_mean_)) / (stats['count'] + self.smoothing_m)
        
        self.actor_means_ = stats['smoothed'].to_dict()
        return self

    def transform(self, X):
        X_new = X.copy()
        
        def get_cast_mean(actors_str):
            if pd.isna(actors_str) or str(actors_str).strip() in ['', 'nan']:
                return self.global_mean_
            actors = [a.strip() for a in str(actors_str).split(',') if a.strip() != '']
            if not actors:
                return self.global_mean_
            
            actor_scores = [self.actor_means_.get(a, self.global_mean_) for a in actors]
            return np.mean(actor_scores)

        X_new['cast_avg_rating'] = X_new['lead_actors_ids'].apply(get_cast_mean)
        # מחיקת עמודת הטקסט המקורית
        X_new = X_new.drop(columns=['lead_actors_ids'])
        return X_new


# --- ב. הכנת הנתונים והפיצול ---
print("מכין את הנתונים ומפצל ל-Train ו-Test...")
processed_df_model = processed_df.dropna(subset=['averageRating']).copy()

y = processed_df_model['averageRating']
X = processed_df_model.drop(columns=['averageRating'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# --- ג. הרכבת ה-Pipeline השלם ---
print("בונה את ה-Pipeline ומתחיל אימון... (זה עשוי לקחת כמה דקות)")
pipeline = Pipeline([
    ('cast_encoder', CastTargetEncoder(min_count=5, smoothing_m=30)), 
    ('poly_features', PolynomialFeaturesGenerator()),
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler()), 
    ('elasticnet', ElasticNet(random_state=42, max_iter=10000)) # 10000 צעדים למניעת שגיאת התכנסות
])

# --- ד. חיפוש פרמטרים ו-Cross Validation ---
param_grid = {
    'elasticnet__alpha': [0.0001, 0.001, 0.01, 0.1, 1.0], 
    'elasticnet__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5, 
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, 
    verbose=1
)

grid_search.fit(X_train, y_train)

# --- ה. תוצאות והערכה ---
print("\n--- תוצאות האימון (Elastic Net Baseline) ---")
print(f"הפרמטרים הטובים ביותר: {grid_search.best_params_}")

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- ביצועים על קבוצת הבדיקה (Test Set) ---")
print(f"RMSE: {rmse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R^2: {r2:.4f}")

מכין את הנתונים ומפצל ל-Train ו-Test...
בונה את ה-Pipeline ומתחיל אימון... (זה עשוי לקחת כמה דקות)
Fitting 5 folds for each of 30 candidates, totalling 150 fits


C:\Users\yarin\.Origin\envs\myenv\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.064e+02, tolerance: 1.549e+01
  model = cd_fast.enet_coordinate_descent(



--- תוצאות האימון (Elastic Net Baseline) ---
הפרמטרים הטובים ביותר: {'elasticnet__alpha': 0.0001, 'elasticnet__l1_ratio': 0.1}

--- ביצועים על קבוצת הבדיקה (Test Set) ---
RMSE: 1.1246
MAE: 0.8606
R^2: 0.2362


In [9]:
# ==========================================
# חלק 2: Random Forest עם מניעת Leakage והנדסת מאפיינים
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor # <--- השינוי המרכזי למודל עצים
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# --- א. יצירת רכיבים מותאמים אישית (Custom Transformers) ---

class PolynomialFeaturesGenerator(BaseEstimator, TransformerMixin):
    """ מוסיף זמן ריצה בריבוע """
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X_new = X.copy()
        if isinstance(X_new, pd.DataFrame) and 'runtimeMinutes' in X_new.columns:
            X_new['runtimeMinutes_sq'] = X_new['runtimeMinutes'] ** 2
        return X_new

class CastTargetEncoder(BaseEstimator, TransformerMixin):
    """
    מקודד שחקנים עם סינון נוקשה (min 5) והחלקה בייסיאנית.
    לומד אך ורק מתוך ה-Train ב-Cross Validation ומונע Data Leakage.
    """
    def __init__(self, min_count=5, smoothing_m=30):
        self.min_count = min_count
        self.smoothing_m = smoothing_m
        self.actor_means_ = {}
        self.global_mean_ = 0

    def fit(self, X, y):
        df_temp = pd.DataFrame({'actors': X['lead_actors_ids'].astype(str), 'rating': y})
        df_temp['actors'] = df_temp['actors'].apply(lambda x: [a.strip() for a in x.split(',') if a.strip() not in ['', 'nan']])
        df_exploded = df_temp.explode('actors').dropna(subset=['actors'])
        
        self.global_mean_ = y.mean()
        
        stats = df_exploded.groupby('actors')['rating'].agg(['mean', 'count'])
        # סינון - רק שחקנים עם לפחות X סרטים ב-Train
        stats = stats[stats['count'] >= self.min_count]
        
        # נוסחת ההחלקה לבלימת Overfitting
        stats['smoothed'] = ((stats['count'] * stats['mean']) + (self.smoothing_m * self.global_mean_)) / (stats['count'] + self.smoothing_m)
        
        self.actor_means_ = stats['smoothed'].to_dict()
        return self

    def transform(self, X):
        X_new = X.copy()
        
        def get_cast_mean(actors_str):
            if pd.isna(actors_str) or str(actors_str).strip() in ['', 'nan']:
                return self.global_mean_
            actors = [a.strip() for a in str(actors_str).split(',') if a.strip() != '']
            if not actors:
                return self.global_mean_
            
            # שליפה מהמילון, או קבלת הממוצע הגלובלי אם לא קיים
            actor_scores = [self.actor_means_.get(a, self.global_mean_) for a in actors]
            return np.mean(actor_scores)

        X_new['cast_avg_rating'] = X_new['lead_actors_ids'].apply(get_cast_mean)
        X_new = X_new.drop(columns=['lead_actors_ids']) # הסרת המזהה המקורי
        return X_new

# --- ב. הכנת הנתונים והפיצול ---
print("מכין את הנתונים ומפצל ל-Train ו-Test...")
processed_df_model = processed_df.dropna(subset=['averageRating']).copy()

y = processed_df_model['averageRating']
X = processed_df_model.drop(columns=['averageRating'])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# --- ג. הרכבת ה-Pipeline של היער האקראי ---
# הערה: עצי החלטה לא באמת צריכים Scaling, אבל זה לא פוגע ונשאר לאחידות הקוד
print("\nמרכיב Pipeline ליער אקראי...")
pipeline_rf = Pipeline([
    ('cast_encoder', CastTargetEncoder(min_count=5, smoothing_m=30)), 
    ('poly_features', PolynomialFeaturesGenerator()),
    ('imputer', SimpleImputer(strategy='median')), 
    ('scaler', StandardScaler()), 
    ('rf', RandomForestRegressor(random_state=42)) 
])

# --- ד. חיפוש פרמטרים ואימון ---
# צמצמתי את החיפוש בכוונה כדי שהריצה לא תיקח נצח, היער הוא אלגוריתם "כבד"
print("\nמתחיל לחפש פרמטרים (Tuning) ולאמן את היער...")
print("שים לב: אימון יער אקראי עם Cross Validation לוקח זמן. אפשר להכין קפה בינתיים \U0001f913")

param_grid_rf = {
    'rf__n_estimators': [100],               # כמות עצים ביער (מספיק 100 ליציבות)
    'rf__max_depth': [20, 30, None],         # עומק העצים - חשוב למניעת Overfitting
    'rf__min_samples_split': [5, 10]         # כמות דגימות מינימלית לפני שחולקים ענף (גם מונע Overfitting)
}

grid_search_rf = GridSearchCV(
    estimator=pipeline_rf,
    param_grid=param_grid_rf,
    cv=10, # הורדתי ל-3 קיפולים כדי להאיץ את התהליך
    scoring='neg_root_mean_squared_error',
    n_jobs=-1, 
    verbose=2 # יציג לך יותר פרטים על ההתקדמות במסך
)

# שעת המבחן!
grid_search_rf.fit(X_train, y_train)

# --- ה. תוצאות והערכה ---
print("\n=== תוצאות האימון (Random Forest) ===")
print(f"הפרמטרים הטובים ביותר: {grid_search_rf.best_params_}")

best_rf_model = grid_search_rf.best_estimator_
y_pred_rf = best_rf_model.predict(X_test)

rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("\n=== ביצועים על קבוצת הבדיקה (Test Set) ===")
print(f"RMSE: {rmse_rf:.4f}")
print(f"MAE: {mae_rf:.4f}")
print(f"R^2: {r2_rf:.4f}")

מכין את הנתונים ומפצל ל-Train ו-Test...

מרכיב Pipeline ליער אקראי...

מתחיל לחפש פרמטרים (Tuning) ולאמן את היער...
שים לב: אימון יער אקראי עם Cross Validation לוקח זמן. אפשר להכין קפה בינתיים 🤓
Fitting 10 folds for each of 6 candidates, totalling 60 fits

=== תוצאות האימון (Random Forest) ===
הפרמטרים הטובים ביותר: {'rf__max_depth': 20, 'rf__min_samples_split': 10, 'rf__n_estimators': 100}

=== ביצועים על קבוצת הבדיקה (Test Set) ===
RMSE: 1.0962
MAE: 0.8322
R^2: 0.2743
